# Autonomous Graph of Thoughts (GoT) Debugging Agent
### Quantized Coding LLM (Qwen2.5-Coder-7B / Llama-3-8B) + Code Property Graph in Google Colab

This notebook runs an autonomous LLM debugging agent powered by **Code Property Graph (CPG)** static analysis.

**Core Mechanism: Graph of Thoughts (GoT)**
- The model **cannot see the whole codebase**.
- Given a failing test symptom, it forms hypotheses, invokes graph traversal tools (`traverse_ast`, `traverse_cfg`, `traverse_data_flow`), merges the observations across syntax, execution paths, and data dependencies, and pinpoints the exact bug and patch.

## Step 1: Workspace Setup (Unpack Project Files)

Google Colab runs on a fresh remote VM. To ensure the scripts (`agent_runner.py`, `agent_tools.py`, etc.) and the extracted `cpg_json/` are available in Colab, choose either:
- **Option A**: Drag and drop `colab_bundle.zip` into the Colab file browser (left panel), then run the cell below.
- **Option B**: Or click the "Choose Files" button generated below to upload `colab_bundle.zip` directly.

In [ ]:
import os, zipfile

if not os.path.exists("cpg_json") and not os.path.exists("agent_runner.py"):
    if os.path.exists("colab_bundle.zip"):
        print("[*] Found colab_bundle.zip! Unpacking...")
        !unzip -q -o colab_bundle.zip
        print("[+] Unpacked successfully!")
    else:
        print("[*] Please upload 'colab_bundle.zip' via the upload button below:")
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith(".zip"):
                !unzip -q -o "$fname"
        print("[+] Project files and cpg_json ready!")
else:
    print("[+] Project files and cpg_json already present in workspace.")

!ls -la

## Step 2: Install GPU Packages
Install PyTorch, HuggingFace Transformers, BitsAndBytes (4-bit quantization), and NetworkX.

In [ ]:
!nvidia-smi
!pip install -q torch transformers bitsandbytes accelerate networkx

## Step 3: Run the Agent Scripts Directly

You can run the agent in two ways:
1. **Command Line Execution**: Run [`agent_runner.py`](file:///home/aaaa/Desktop/Projects/GDAC/agent_runner.py) directly with Python flags.
2. **Interactive Python Execution**: Import and interact with the model and tools step-by-step.

### Quick Test: Verify Graph Traversals & Dry Run (No GPU Wait)

In [ ]:
# Verifies that cpg_json and the three graph tools work seamlessly
!python agent_runner.py --cpg-path cpg_json --dry-run --target discount

## Step 4: Load 4-bit Quantized Model (Qwen2.5-Coder-7B or Llama-3-8B)
Loads the model into GPU VRAM using 4-bit NormalFloat (NF4) quantization (~5.5 GB VRAM).

In [ ]:
from agent_runner import load_quantized_model

# Loads Qwen2.5-Coder-7B-Instruct in 4-bit
model, tokenizer = load_quantized_model(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    quantization="4bit"
)

## Step 5: Execute the Autonomous Graph of Thoughts Agent Loop
The agent will:
1. Read the test failure report.
2. Formulate a hypothesis.
3. Call `traverse_ast` to locate the method and its parameters.
4. Call `traverse_cfg` to trace runtime execution branches.
5. Call `traverse_data_flow` to trace variable mutations.
6. Output the exact root cause and code patch.

### Bug Scenario 1: Discount Calculation Defect (`apply_discount`)

In [ ]:
from cpg_bridge import load_cpg_json
from agent_prompt import format_user_problem_prompt
from agent_runner import run_agent_loop

graph = load_cpg_json("cpg_json")

problem_prompt_1 = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_inventory.py::TestInventoryManager::test_apply_discount\n"
        "AssertionError: 20.0 != 80.0\n"
        "Applying a 20% discount on $100 item resulted in $20.00 instead of $80.00."
    ),
    bug_description="Unit test fails when calculating percentage discounts."
)

deduction_1 = run_agent_loop(model, tokenizer, graph, problem_prompt_1, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(deduction_1)

### Bug Scenario 2: Category Summary Aggregation Defect (`generate_category_summary`)

In [ ]:
problem_prompt_2 = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_reports.py::TestInventoryReporter::test_generate_category_summary_aggregates_multiple_items\n"
        "AssertionError: 250.0 != 1000.0\n"
        "Category summary only shows the last product value ($250) instead of aggregate total ($1000)."
    ),
    bug_description="Category valuation summary ignores multiple items in the same category."
)

deduction_2 = run_agent_loop(model, tokenizer, graph, problem_prompt_2, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(deduction_2)